# Loading libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.impute import SimpleImputer
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from collections import Counter
from pathlib import Path
import joblib


In [ ]:
RESULTS_DIR = Path("results")

# Metrics plots

In [ ]:
metrics = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/results/metrics/SupplementaryTable28.csv")

In [ ]:
dataset_order = ["v1", "v2", "v3"]
model_order = metrics["estimator"].unique()

metrics_map = {
    "recall_at_01_pct": "Recall@0.1",
    "recall_at_05_pct": "Recall@0.5",
    "recall_at_10_pct": "Recall@1.0",
    "pr_auc": "PR-AUC",
    "execution_time_sec": "Execution time (s)"
}

metrics = metrics.rename(columns=metrics_map)

metrics_order = [
    "Recall@0.1",
    "Recall@0.5",
    "Recall@1.0",
    "PR-AUC",
    "Execution time (s)"
]

sns.set_theme(style="white", context="talk")

fig, axes = plt.subplots(
    2,
    3,
    figsize=(18, 10)
)

axes = axes.flatten()

for i, metric in enumerate(metrics_order):

    ax = axes[i]

    sns.barplot(
        data=metrics,
        x="dataset",
        y=metric,
        hue="estimator",
        order=dataset_order,
        hue_order=model_order,
        ax=ax
    )

    ax.set_title(metric, fontsize=14, weight="bold", pad=15)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.grid(False)

    if ax.get_legend() is not None:
        ax.get_legend().remove()

    for spine in ax.spines.values():
        spine.set_linewidth(1.3)

    ax.tick_params(
        direction="out",
        length=5,
        width=1.3,
        top=True,
        right=True
    )

handles, labels = axes[0].get_legend_handles_labels()

axes[5].axis("off")

axes[5].legend(
    handles,
    labels,
    title="Model",
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    frameon=False,
    fontsize=16,
    title_fontsize=18,
    ncol=1
)

plt.tight_layout()



plt.savefig(
    RESULTS_DIR / "plots/metrics_grid.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    RESULTS_DIR / "plots/metrics_grid.svg",
    bbox_inches="tight"
)

plt.show()

# Priorisation analysis

In [ ]:
best_rf = joblib.load(RESULTS_DIR / "models/final_tuned_pu_rf_model.pkl")

In [ ]:
ml_dataset = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/datasets_final/ml_dataset.csv")

In [ ]:
discovery_df = ml_dataset[ml_dataset["label"] != 1].copy()

metadata_cols = ["label", "User_input"]

num_cols = [
    c for c in discovery_df.select_dtypes(include=[np.number]).columns
    if c not in metadata_cols
]

X_discovery = discovery_df[num_cols].copy()

imputer = joblib.load(
    RESULTS_DIR / "models/median_imputer.pkl"
)

X_discovery = pd.DataFrame(
    imputer.transform(X_discovery),
    columns=num_cols,
    index=discovery_df.index
)

probs = best_rf.predict_proba(X_discovery)[:, 1]

discovery_df["predicted_probability"] = probs

print(type(best_rf))
print(best_rf)

print(best_rf.__dict__.keys())


In [ ]:
def selecting_top_x(x, discovery_df):
  top_x = discovery_df.nlargest(x, "predicted_probability")
  X_top_x = X_discovery.loc[top_x.index]

  remaining = discovery_df.drop(top_x.index)
  X_remaining = X_discovery.loc[remaining.index]

  return top_x, X_top_x, remaining, X_remaining




def shap_analysis(X_top_x, best_rf, x):
  all_shap = []

  for rf in best_rf.estimators_:

    explainer = shap.TreeExplainer(rf)

    shap_values = explainer.shap_values(X_top_x)

    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    else:
        shap_values = shap_values[:, :, 1]

    all_shap.append(shap_values)

  mean_shap = np.mean(all_shap, axis=0)



  importance = pd.DataFrame({
      "Feature": X_discovery.columns,
      "Mean_SHAP": np.abs(mean_shap).mean(axis=0)
  })

  importance = importance.sort_values(
      "Mean_SHAP",
      ascending=False
  )

  important_features = importance.head(10)["Feature"]



  shap.summary_plot(
    mean_shap,
    X_top_x,
    feature_names=X_top_x.columns,
    show=False
)

  fig = plt.gcf()
  fig.savefig(
      RESULTS_DIR / f"plots/shap_summary_plot_top{x}.png",
      dpi=300,
      bbox_inches="tight"
  )

  fig.savefig(
      RESULTS_DIR / f"plots/shap_summary_plot_top{x}.svg",
      dpi=300,
      bbox_inches="tight"
  )

  plt.show()
  plt.close(fig)

  return important_features, mean_shap




def build_boxplots(important_features, X_top_x, X_remaining, x):
    with sns.axes_style("whitegrid"), sns.plotting_context("talk"):
        median_df = pd.DataFrame(columns=["feature", "median_top", "median_remaining"])

        for feature in important_features:

            plot_df = pd.DataFrame({
                "Value": pd.concat([
                    X_top_x[feature],
                    X_remaining[feature]
                ], ignore_index=True),
                "Group": ([f"Top {x}"] * len(X_top_x)) +
                        (["Remaining"] * len(X_remaining))
            })

            median_top = X_top_x[feature].median()
            median_remaining = X_remaining[feature].median()
            median_df_feature = pd.DataFrame({
                "feature": [feature],
                "median_top": [median_top],
                "median_remaining": [median_remaining],
                "top": [x],
            })
            median_df = pd.concat([median_df, median_df_feature], ignore_index=True)
            median_df.to_csv(RESULTS_DIR / f"metrics/medians_feature_top{x}.csv", index=False)

            plot_df = pd.concat([plot_df, median_df], ignore_index=True)

            fig, ax = plt.subplots(figsize=(4, 5))

            sns.boxplot(
                data=plot_df,
                x="Group",
                y="Value",
                palette=["#D55E00", "#0072B2"],
                width=0.4,
                fliersize=0,
                linewidth=1.5,
                ax=ax
            )

            ax.set_title(feature, fontsize=14, weight="bold")
            ax.set_xlabel("")
            ax.set_ylabel("")

            sns.despine()

            plt.tight_layout()

            fig.savefig(
                RESULTS_DIR / f"plots/{feature}_boxplot_top{x}.png",
                dpi=300,
                bbox_inches="tight"
            )

            fig.savefig(
                RESULTS_DIR / f"plots/{feature}_boxplot_top{x}.svg",
                bbox_inches="tight"
            )

            plt.show()
            plt.close(fig)




def akna_variant_comparison_remaining(discovery_df, X_discovery, important_features, X_top_x, best_rf):
  AKNA_ID = "9:114341693:A:G:AKNA"

  top20 = discovery_df.nlargest(20, "predicted_probability")

  if AKNA_ID not in top20["User_input"].values:
      top20 = pd.concat([
          discovery_df.loc[
              discovery_df["User_input"] == AKNA_ID
          ],
          top20
      ]).drop_duplicates("User_input")

  comparison = pd.concat(
      [
          top20[
              [
                  "User_input",
                  "predicted_probability",
              ]
            ],
          X_discovery.loc[
              top20.index,
              important_features
          ]
      ],
      axis=1
  )

  comparison.to_csv(RESULTS_DIR / "metrics/akna_comparison_top-20.csv", index=False)


  akna_rows = discovery_df[
    discovery_df["User_input"] == AKNA_ID
    ]

  if akna_rows.empty:
      raise ValueError(f"{AKNA_ID} not found in discovery dataset.")

  akna_idx = akna_rows.index[0]

  if akna_idx not in X_top_x.index:
      print(f"AKNA is not present in the Top {len(X_top_x)} variants.")
      return

  akna_pos = X_top_x.index.get_loc(akna_idx)


  all_shap = []
  all_base = []

  for rf in best_rf.estimators_:

    explainer = shap.TreeExplainer(rf)

    shap_values = explainer.shap_values(X_top_x)

    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    else:
        shap_values = shap_values[:, :, 1]

    all_shap.append(shap_values)

    if isinstance(explainer.expected_value, (list, np.ndarray)):
        all_base.append(explainer.expected_value[1])
    else:
        all_base.append(explainer.expected_value)

  mean_shap = np.mean(all_shap, axis=0)
  mean_base = np.mean(all_base)

  shap.plots.waterfall(
      shap.Explanation(
        values=mean_shap[akna_pos],
        base_values=mean_base,
        data=X_top_x.iloc[akna_pos],
        feature_names=X_top_x.columns
    ),
    max_display=15,
    show=False
  )

  fig = plt.gcf()
  fig.savefig(
      RESULTS_DIR / "plots/akna_variant_shap_waterfall.png",
      dpi=300,
      bbox_inches="tight"
  )

  fig.savefig(
      RESULTS_DIR / "plots/akna_variant_shap_waterfall.svg",
      dpi=300,
      bbox_inches="tight"
  )

  plt.show()
  plt.close(fig)


def pca_analysis(X_top_x, top_x, x):
  AKNA_ID = "9:114341693:A:G:AKNA"

  X_scaled = StandardScaler().fit_transform(X_top_x)

  pca = PCA(n_components=2, random_state=42)
  pcs = pca.fit_transform(X_scaled)

  kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
  clusters = kmeans.fit_predict(X_scaled)

  pca_df = pd.DataFrame(
      {
          "PC1": pcs[:, 0],
          "PC2": pcs[:, 1],
          "Cluster": clusters,
          "predicted_probability": top_x["predicted_probability"].values,
      },
      index=top_x.index,
  )

  metadata = [
      col
      for col in ["User_input"]
      if col in top_x.columns
  ]

  pca_df = pca_df.join(top_x[metadata])

  pca_df.to_csv(RESULTS_DIR / f"metrics/pca_top{x}.csv")


  with sns.plotting_context("talk"), sns.axes_style("white"):

    fig, ax = plt.subplots(figsize=(8, 6))

    sns.scatterplot(
        data=pca_df,
        x="PC1",
        y="PC2",
        hue="Cluster",
        size="predicted_probability",
        palette="Dark2",
        ax=ax
    )

    ax.grid(False)


  if "User_input" in pca_df.columns:
      akna = pca_df[pca_df["User_input"] == AKNA_ID]
      for _, row in akna.iterrows():
          ax.annotate(
              "9:114341693:A:G:AKNA",
              xy=(row["PC1"], row["PC2"]),
              xytext=(8, 5),
              textcoords="offset points",
              fontsize=6,
              ha="left",
              va="bottom"
          )

  ax.legend(
      bbox_to_anchor=(1.05, 1),
      loc="upper left",
      borderaxespad=0.
  )

  plt.savefig(
      RESULTS_DIR / f"plots/pca_top{x}.png",
      dpi=300,
      bbox_inches="tight"
      )

  plt.savefig(
      RESULTS_DIR / f"plots/pca_top{x}.svg",
      dpi=300,
      bbox_inches="tight"
      )

  plt.show()


In [ ]:
x = 100

topx, X_topx, remainingx, X_remainingx = selecting_top_x(x, discovery_df)

important_featuresx, mean_shapx = shap_analysis(X_topx, best_rf, x)

build_boxplots(
    important_featuresx,
    X_topx,
    X_remainingx,
    x
)

comparisonx = akna_variant_comparison_remaining(
    discovery_df,
    X_discovery,
    important_featuresx,
    X_topx,
    best_rf
)

pca_dfx = pca_analysis(X_topx, topx, x)

# Similarity analysis

In [ ]:


model = joblib.load(RESULTS_DIR / "models/final_tuned_pu_rf_model.pkl")

full_dataset = pd.read_csv(
    "/Users/ricardo/Desktop/MECD/2ndYear/Thesis/datasets_final/ml_dataset.csv"
)

pgx = full_dataset[
    full_dataset["label"] == 1
].copy()


metadata = ["label", "User_input"]

feature_cols = [
    c for c in pgx.columns
    if c not in metadata
]


imputer = joblib.load(
    RESULTS_DIR / "models/median_imputer.pkl"
)

X_pgx = pd.DataFrame(
    imputer.transform(pgx[feature_cols]),
    columns=feature_cols,
    index=pgx.index
)


discovery = full_dataset[
    full_dataset["label"] != 1
].copy()

X_discovery = pd.DataFrame(
    imputer.transform(discovery[feature_cols]),
    columns=feature_cols,
    index=discovery.index
)

## Select AKNA

AKNA_ID = "9:114341693:A:G:AKNA"

akna = X_discovery.loc[
    discovery["User_input"] == AKNA_ID
]


## RF proximity

n_reference = len(X_pgx)

proximity = np.zeros(n_reference)

total_trees = 0

for rf in model.estimators_:

    for tree in rf.estimators_:

        total_trees += 1

        akna_leaf = tree.apply(akna)[0]

        pgx_leaf = tree.apply(X_pgx)

        proximity += (pgx_leaf == akna_leaf)

proximity /= total_trees


## Results

results = pgx.copy()

results["RF_proximity"] = proximity


# Create AKNA row

akna_row = discovery.loc[
    discovery["User_input"] == AKNA_ID
].copy()

akna_row["RF_proximity"] = 1.0

akna_row["Gene"] = "AKNA"
akna_row["label"] = "Discovery"

# Combine

results = pd.concat(
    [akna_row[results.columns], results],
    ignore_index=True
)

results = results.sort_values(
    "RF_proximity",
    ascending=False
).reset_index(drop=True)


labelled_dataset = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/datasets_final/final_clinpgx_processed_dataset.csv",
                               sep="\t")
labelled_dataset = labelled_dataset[['User_input', 'Drug(s)', 'Phenotype Category', 'Is/Is Not associated', 'Direction of effect',
                                    'PD/PK terms', 'Population types', 'Population Phenotypes or diseases', 'Alleles',
                                     'Comparison Allele(s) or Genotype(s)']]


results_pgx = pd.merge(results[['User_input', 'RF_proximity']], labelled_dataset, on='User_input', how='left')


results.to_csv(
    RESULTS_DIR / "metrics/akna_rf_neighbours.csv",
    index=False
)

results_pgx.to_csv(
    RESULTS_DIR / "metrics/akna_rf_neighbours_withpgx.csv",
    index=False
)


In [ ]:

# AKNA raw features
akna_raw = full_dataset.loc[
    full_dataset["User_input"] == AKNA_ID
].copy()


# Labelled PGx variants
pgx = full_dataset.loc[
    full_dataset["label"] == 1
].copy()


comparison = pd.concat(
    [akna_raw, pgx],
    ignore_index=True
)


X = comparison[feature_cols]


## Median imputation


X = pd.DataFrame(
    imputer.transform(X),
    columns=feature_cols,
    index=comparison["User_input"]
)


# RF proximity
# (results already computed in previous cell)
top = (
    results.loc[
        results["User_input"] != AKNA_ID
    ]
    .copy()
)

## Missingness analysis

all_lookup = full_dataset.set_index("User_input")

akna_raw = all_lookup.loc[AKNA_ID]

akna_missing = {
    col
    for col in feature_cols
    if pd.isna(akna_raw[col])
}


missingness = []

missingness.append({
    "User_input": AKNA_ID,
    "RF_proximity": 1.0,
    "n_imputed": len(akna_missing),
    "shared_imputed_features": len(akna_missing),
    "shared_features": ", ".join(sorted(akna_missing))
})

for _, row in top.iterrows():

    variant = all_lookup.loc[row["User_input"]]

    missing = {
        col
        for col in feature_cols
        if pd.isna(variant[col])
    }

    missingness.append({
        "User_input": row["User_input"],
        "RF_proximity": row["RF_proximity"],
        "n_imputed": len(missing),
        "shared_imputed_features": len(akna_missing & missing),
        "shared_features": ", ".join(sorted(akna_missing & missing))
    })

missingness = pd.DataFrame(missingness)

missingness.to_csv(
    RESULTS_DIR /
    "metrics/akna_neighbours_missingness.csv",
    index=False
)


## Standardize ALL discovery variants
scaler = StandardScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=feature_cols,
    index=comparison["User_input"]
)

akna = X_scaled.loc[AKNA_ID]


## Feature similarity
similarity = []

for _, row in top.iterrows():

    neighbour = row["User_input"]

    if neighbour not in X_scaled.index:
        continue

    diff = (
        X_scaled.loc[neighbour] - akna
    ).abs()

    closest = diff.nsmallest(10)

    similarity.append({

        "User_input":
            neighbour,

        "RF_proximity":
            row["RF_proximity"],

        "mean_difference":
            diff.mean(),

        "median_difference":
            diff.median(),

        "max_difference":
            diff.max(),

        "n_features_difference<0.25":
            (diff < 0.25).sum(),

        "n_features_difference<0.50":
            (diff < 0.50).sum(),

        "most_similar_features":
            ", ".join(closest.index)

    })

similarity = (
    pd.DataFrame(similarity)
    .sort_values(
        "RF_proximity",
        ascending=False
    )
)

similarity.to_csv(
    RESULTS_DIR /
    "metrics/akna_feature_similarity.csv",
    index=False
)


## Which features appear most often?
counter = Counter()

for features in similarity["most_similar_features"]:

    counter.update(
        features.split(", ")
    )

feature_frequency = (
    pd.DataFrame(
        counter.items(),
        columns=[
            "Feature",
            "Frequency"
        ]
    )
    .sort_values(
        "Frequency",
        ascending=False
    )
)

feature_frequency.to_csv(
    RESULTS_DIR /
    "metrics/akna_similarity_feature_frequency.csv",
    index=False
)




plt.figure(figsize=(8,6))

feature_frequency.head(20).sort_values(
    "Frequency"
).plot.barh(
    x="Feature",
    y="Frequency",
    legend=False
)

plt.xlabel("Frequency among top-10 similar features")
plt.ylabel("")
plt.tight_layout()

plt.savefig(
    RESULTS_DIR /
    "plots/akna_similarity_feature_frequency.png",
    dpi=300
)

plt.show()